In [ ]:
# Main working folder

# Task 1: File ingestion and regular expressions

# 1. write a robust file reader function
import re
from pathlib import Path


def read_file(file_path: Path) -> tuple[str, str]:
    """
    Function reads files, and returns a tuple that 
    contains both the content of the file (as strings), as 
    well as the encoding that is used. The function is able to 
    handle both UTF-8 and iso-8859-1 encoding. 
    
    Args: Path object that points the file that is supposed
    to be read. 
    Returns: text file, encoding
    """
    try: 
        text = file_path.read_text(encoding="utf-8")
        return text, "utf-8"
    except UnicodeDecodeError:
        text = file_path.read_text(encoding="iso-8859-1")
        return text, "iso-8859-1"


# 2. write a log parser function


def parser_func(log_content: str) -> list[dict]:
    """
    Function looks through lines, finding timestamp, register_id, 
    item, price, currency, and status while ignoring
    noice lines. Function looks for square brackets (/[, /]),
    creates labeled slots with the appropriate name (?P<"name">),
    and pulls out the item inside (.*?) Everything is collected
    in a dictorinary. The items are put into the correct format
    according to what they are. 

    Args: 
    Returns: 
    """

    valid_lines = (
        r"\[(?P<timestamp>.*?)\] "  
      r"\[(?P<register_id>.*?)\] "  
      r"item=(?P<item>\w+) " # words with one or more letters/numbers
      r"price=(?P<price>[\d.]+) " # digits, one or more
      r"(?P<currency>[A-Z]{3}) " # big letters from A-Z, three letters. "EUR" as currency
      r"\((?P<status>\w+)\)"
    )


    transaction_record = []

    # turn groups above into dictionaries before
    # adding to transaction record
    for match in re.finditer(valid_lines, log_content):
        convert_dictionary = match.groupdict()
        convert_dictionary["price"]=float(convert_dictionary["price"])
        transaction_record.append(convert_dictionary)

    return transaction_record

# Testing the code
test = Path("Data/2026/library_cafe/POS_01/register_sales.log")

text, code = read_file(test)
transactions = parser_func(text)

print(f"Encoding: {code}")
print(f"Amount of transactions: {len(transactions)}")
print(f"First transaction: {transactions}")
  



In [10]:
# Task 2: Recursive utility functions 

import json
from pathlib import Path

# 1. resursive directory search function
def find_files(directory: str|Path) -> list[Path]:
    """
    Function finds all log files in a nested directionary. Uses Path.iterdir()
    to go through items. 
    Path.glob(), Path.rglob(), os.walk() are not used. 

    Args: 
    Returns: list of pathlib.Path objects.
    """

    dir = Path(directory)
    log_file = []

    if dir.is_dir():
        for item in dir.iterdir():
            if item.is_dir(): 
                log_file.extend(find_files(item))
            elif item.is_file() and item.suffix == ".log":
                log_file.append(item)

    return log_file

# 2. Combination of nested dictionaries and lists. Returns register id strings. 
def active_register_id(data: dict|list) -> set[str]:
    """
    Function goes through dictionaries and lists found in store_config.json. 
    Function finds all registered id strings that are active. 
    
    Args: 
    Returns: a set of all resgister id strings that are active. 
    """

    active_register=set()
    if isinstance(data, dict): #if data is a dictionary

        if "id" in data and data.get("active") is True:
            active_register.add(str(data["id"]))
        for value in data.values():
           active_register.update(active_register_id(value))
    elif isinstance(data, list): #if data is a list
        for item in data: 
            active_register.update(active_register_id(item))

    return active_register


# Testing the code: 
file = list(Path(".").rglob("store_config.json"))
if not file:
  file = list(Path("..").rglob("store_config.json"))

if file:
  config_file = file[0]
  test_folder = config_file.parent

  # Task 2.1 Test
  log_file = find_files(test_folder)
  print(f"Amount of .log-filer: {len(log_file)}")
  if log_file:
    print(f"First log-file: {log_file[0]}")

  # Task 2.2 Test
  with open(config_file) as f:
    json_data = json.load(f)

  active = active_register_id(json_data)
  print(f"Active registers: {active}")
  print(f"Amount of active registers: {len(active)}")

  

Amount of .log-filer: 0
Active registers: {'POS_02', 'POS_01', 'POS_03'}
Amount of active registers: 3


In [11]:
# Task 3: Functions for cleaning and analysing data using NumPy

import numpy as np

# 1. function for removing price outliers 

def price_outliers(price: list[float]| np.ndarray) -> np.ndarray: 

    """
    Function removes price outliers. Price is put into a 1D NumPy array and
    outliers were value deviates from the mean by more than 3 standard deviations
    are removed and replaced with the median of valid prices. 

    Args: 
    Returns: numpy array without outliers. 
    """
    price = np.array(price, dtype = float) # converts into numpy array

    mean = np.mean(price) #mean
    sd = np.std(price) #standard deviation

    valid_prices = []
    for p in price:
        if np.abs(price-mean) <= 3 * sd: 
            valid_prices.append(p)

    median = np.median(valid_prices) #median 
    outliers = np.abs(price-mean) > 3 * sd
    price[outliers] = median #replace outliers with median

    return price


# 2. function that computes ratio and addition of epsilon


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Function computes element-wise refund ratio and adds epsilon to prevent zero-division 
    errors. 

    Args: 
    Returns: Refund ratio for each item as numpy array. 
    """
    
    epsilon = 1e-6
    ratios = []

    for i in range(len(sales_count)): 
        sale = sales_count[i]
        refund = refund_count[i]

        ratio = refund / (sale + epsilon)
        ratios.append(ratio)

    return np.array(ratios)



In [12]:
# Task 4: DataFrame manipulation with Pandas

import pandas as pd
import numpy as np

# Task 4.1: Convert transactions to a DataFrame

def create_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Converts a list of transaction dictionaries into a DataFrame.
    Converts timestamp to datetime, sets it as the index,
    and sorts the transactions chronologically.
    """
    df = pd.DataFrame(transactions)

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()

    return df


# Task 4.2: Calculate hourly revenue per item

def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """
    Groups transactions by item and calculates the total price
    in one-hour intervals.
    """
    hourly = (
        df.groupby("item")
        .resample("1h")["price"]
        .sum()
        .reset_index()
    )

    return hourly


# Task 4.3: Create a pivot table by item and status

def create_status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates a pivot table showing total transaction revenue
    grouped by item and status.
    """
    pivot = pd.pivot_table(
        df,
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0
    )

    pivot = pivot.reindex(
        columns=["SUCCESS", "REFUND", "CANCELLED"],
        fill_value=0
    )

    return pivot


# Task 4.4: Calculate refund ratio per item using Task 3

def refund_ratio(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates the refund ratio for each item by calling
    compute_refund_ratio() from Task 3.
    """

    # Count transactions for each item and status
    counts = pd.crosstab(df["item"], df["status"])

    # Ensure both columns exist, even if a status is missing
    counts = counts.reindex(
        columns=["SUCCESS", "REFUND"],
        fill_value=0
    )

    # Convert counts to NumPy arrays
    sales_count = counts["SUCCESS"].to_numpy()
    refund_count = counts["REFUND"].to_numpy()

    # Call the function already defined in Task 3
    ratios = compute_refund_ratio(sales_count, refund_count)

    # Create the result DataFrame
    result = pd.DataFrame({
        "item": counts.index,
        "refund_ratio": ratios
    })

    return result


# Testing Task 04

test_transactions = [
    {"timestamp": "2026-01-01 10:15:00", "item": "Coffee", "price": 30.0, "status": "SUCCESS"},
    {"timestamp": "2026-01-01 10:30:00", "item": "Coffee", "price": 30.0, "status": "SUCCESS"},
    {"timestamp": "2026-01-01 11:00:00", "item": "Coffee", "price": 30.0, "status": "REFUND"},
    {"timestamp": "2026-01-01 11:20:00", "item": "Tea", "price": 25.0, "status": "SUCCESS"}
]

test_df = create_dataframe(test_transactions)

print("Task 4.1: DataFrame")
display(test_df)

print("Task 4.2: Hourly revenue")
display(hourly_revenue(test_df))

print("Task 4.3: Status pivot table")
display(create_status_pivot(test_df))

print("Task 4.4: Refund ratio")
display(refund_ratio(test_df))

Task 4.1: DataFrame


,item,price,status
timestamp,,,
2026-01-01 10:15:00,Coffee,30.0,SUCCESS
2026-01-01 10:30:00,Coffee,30.0,SUCCESS
2026-01-01 11:00:00,Coffee,30.0,REFUND
2026-01-01 11:20:00,Tea,25.0,SUCCESS


Task 4.2: Hourly revenue


,item,timestamp,price
0,Coffee,2026-01-01 10:00:00,60.0
1,Coffee,2026-01-01 11:00:00,30.0
2,Tea,2026-01-01 11:00:00,25.0


Task 4.3: Status pivot table


status,SUCCESS,REFUND,CANCELLED
item,,,
Coffee,60.0,30.0,0
Tea,25.0,0.0,0


Task 4.4: Refund ratio


,item,refund_ratio
0,Coffee,0.5
1,Tea,0.0


In [23]:
# Task 5: Create full workflow

from pathlib import Path
import json
import numpy as np
import pandas as pd

# Step 1: Find all log files
data_folder = Path("Data")

log_files = find_files(data_folder)
print("Number of log files:", len(log_files))


# Step 2: Find active register IDs
config_files = list(data_folder.rglob("store_config.json"))

if not config_files:
    raise FileNotFoundError("Could not find store_config.json")

active_registers = set()

for config_file in config_files:
    with open(config_file, encoding="utf-8") as f:
        config_data = json.load(f)

    active_registers.update(active_register_id(config_data))

print("Active registers:", active_registers)


# Step 3: Read and parse all log files
all_transactions = []

for log_file in log_files:
    text, encoding = read_file(log_file)
    transactions = parser_func(text)

    for transaction in transactions:
        if transaction["register_id"] in active_registers:
            all_transactions.append(transaction)

print("Number of transactions:", len(all_transactions))

if not all_transactions:
    raise ValueError("No transactions found for active registers")


# Step 4: Create the initial DataFrame
df = create_dataframe(all_transactions)

print("\nInitial DataFrame:")
display(df.head())


# Step 5: Clean price outliers using Task 3
def price_outliers(prices):
    prices = np.asarray(prices, dtype=float)

    mean = np.mean(prices)
    sd = np.std(prices)

    valid_mask = np.abs(prices - mean) <= 3 * sd
    valid_prices = prices[valid_mask]

    median = np.median(valid_prices)

    cleaned_prices = prices.copy()
    cleaned_prices[~valid_mask] = median

    return cleaned_prices
df["price"] = price_outliers(df["price"].to_numpy())

print("\nDataFrame after outlier cleaning:")
display(df.head())


# Step 6: Correct NOK prices
df.loc[df["currency"] == "NOK", "price"] *= 10.0

print("\nDataFrame after currency correction:")
display(df.head())


# Step 7: Generate reports
hourly_report = hourly_revenue(df)
status_pivot = create_status_pivot(df)
refund_report = refund_ratio(df.reset_index(drop = True))

print("\nHourly revenue:")
display(hourly_report.head())

print("\nStatus pivot table:")
display(status_pivot)

print("\nRefund ratios:")
display(refund_report)


# Step 8: Save output files

output_folder = Path("output")
output_folder.mkdir(exist_ok = True)

# Save cleaned transactions as CSV
df.to_csv(output_folder / "cleaned_transactions.csv")

# Save hourly revenue as JSON
hourly_report.to_json(
    output_folder / "hourly_revenue.json",
    orient="records",
    indent=4,
    date_format="iso"
)

# Save status pivot table as CSV
status_pivot.to_csv(output_folder / "status_pivot_table.csv")

# Save refund ratios as CSV
refund_report.to_csv(
    output_folder / "item_refund_ratios.csv",
    index=False
)

print("\nTask 5 workflow completed!")
print("Output files saved in:", output_folder)

Number of log files: 9
Active registers: {'POS_02', 'POS_01', 'POS_03'}
Number of transactions: 720

Initial DataFrame:


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:11,POS_03,Sandwich,7.16,NOK,SUCCESS
2026-10-15 07:00:11,POS_03,Latte,4.73,USD,REFUND
2026-10-15 07:00:12,POS_01,Latte,4.61,NOK,REFUND
2026-10-15 07:00:30,POS_01,Sandwich,999.00,USD,SUCCESS
2026-10-15 07:01:08,POS_03,Espresso,3.17,NOK,SUCCESS



DataFrame after outlier cleaning:


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:11,POS_03,Sandwich,7.16,NOK,SUCCESS
2026-10-15 07:00:11,POS_03,Latte,4.73,USD,REFUND
2026-10-15 07:00:12,POS_01,Latte,4.61,NOK,REFUND
2026-10-15 07:00:30,POS_01,Sandwich,4.02,USD,SUCCESS
2026-10-15 07:01:08,POS_03,Espresso,3.17,NOK,SUCCESS



DataFrame after currency correction:


...


Status pivot table:


status,SUCCESS,REFUND,CANCELLED
item,,,
Cappuccino,1631.37,405.18,400.91
Croissant,1181.42,500.27,264.67
Espresso,936.72,201.86,163.52
Latte,1843.34,423.53,188.04
Muffin,785.92,231.95,469.42
Sandwich,2780.40,529.58,404.23



Refund ratios:


,item,refund_ratio
0,Cappuccino,0.272727
1,Croissant,0.351351
2,Espresso,0.157303
3,Latte,0.302632
4,Muffin,0.151899
5,Sandwich,0.190476



Task 5 workflow completed!
Output files saved in: output
